In [1]:
import os, sys, glob, json, re, time, subprocess
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"; os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"; os.environ["HF_HUB_DISABLE_XET"] = "1"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "language_tool_python", "pyspellchecker", "textstat", "gradio"])

import numpy as np, pandas as pd, torch, torch.nn as nn, lightgbm as lgb, spacy
from collections import Counter
from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoModel, AutoModelForSequenceClassification, AutoTokenizer
from spellchecker import SpellChecker
import textstat, language_tool_python

def find(name, many=False):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"{name} not found - check the attached datasets (aes_dataset must be version 1)"
    return hits if many else hits[0]

# ---------- configs and LightGBM models ----------
aes_conf = json.load(open(find("aes_blend_config.json")))
ell_conf = json.load(open(find("ell_blend_config.json")))
AES_FEATS, W, CUTS = aes_conf["feature_cols"], aes_conf["deberta_weight"], aes_conf["cuts"]
ELL_FEATS = ell_conf["feature_cols"]
TRAITS = ["cohesion", "syntax", "vocabulary", "phraseology", "grammar", "conventions"]
aes_booster  = lgb.Booster(model_file=find("lgbm_aes_features.txt"))
ell_boosters = {t: lgb.Booster(model_file=find(f"lgbm_ell_{t}.txt")) for t in TRAITS}

# ---------- tokenizer and neural models ----------
path = snapshot_download("microsoft/deberta-v3-base", allow_patterns=["*.json", "spm.model"])
tok = AutoTokenizer.from_pretrained(path)
assert (tok.pad_token_id, tok.cls_token_id, tok.sep_token_id) == (0, 1, 2)
CLS, SEP = 1, 2
device = torch.device("cuda:0")

def wrap(ids, maxlen=512):
    room = maxlen - 2
    if len(ids) > room:
        head = int(room * 0.75)
        ids = ids[:head] + ids[len(ids) - (room - head):]
    return [CLS] + ids + [SEP]

def tensors(batch_ids):
    L = max(len(x) for x in batch_ids)
    ids = torch.zeros((len(batch_ids), L), dtype=torch.long); att = torch.zeros_like(ids)
    for r, x in enumerate(batch_ids):
        ids[r, :len(x)] = torch.tensor(x); att[r, :len(x)] = 1
    return ids.to(device), att.to(device)

class Scorer(nn.Module):
    def __init__(self):
        super().__init__()
        c = AutoConfig.from_pretrained(path)
        self.backbone = AutoModel.from_config(c)
        self.head = nn.Linear(c.hidden_size, 1)
    def forward(self, ids, att):
        h = self.backbone(input_ids=ids, attention_mask=att).last_hidden_state.float()
        m = att.unsqueeze(-1).float()
        return self.head((h * m).sum(1) / m.sum(1).clamp(min=1))

essay_models = []
for f in find("aes_fold*.pt", many=True):
    m = Scorer().to(device).eval()
    m.load_state_dict({k: v.float() for k, v in torch.load(f, map_location="cpu").items()})
    essay_models.append(m)
cola_model = AutoModelForSequenceClassification.from_config(AutoConfig.from_pretrained(path, num_labels=2)).to(device).eval()
cola_model.load_state_dict({k: v.float() for k, v in torch.load(find("cola_deberta_best.pt"), map_location="cpu").items()})
print("DeBERTa essay models:", len(essay_models), "| CoLA model loaded")

# ---------- text cleaning (same as in training) ----------
MOJIBAKE = {"â\x80\x99": "'", "â\x80\x98": "'", "â\x80\x9c": '"', "â\x80\x9d": '"',
            "â\x80\x94": "—", "â\x80\x93": "–", "â\x80¦": "…"}
C1 = {}
for code in range(0x80, 0xA0):
    try: C1[code] = bytes([code]).decode("cp1252")
    except UnicodeDecodeError: pass
QUOTES = {ord("’"): "'", ord("‘"): "'", ord("“"): '"', ord("”"): '"'}
def normalize(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()
def fix_text(t):
    for bad, good in MOJIBAKE.items(): t = t.replace(bad, good)
    return t.translate(C1).translate(QUOTES)

# ---------- surface + vocabulary features ----------
WORD_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")
SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")
LOWER_I_RE = re.compile(r"\bi\b")
def surface_lexical_features(text):
    words = WORD_RE.findall(text); n_words = max(len(words), 1); lower = [w.lower() for w in words]
    sents = [s for s in SENT_SPLIT.split(text.strip()) if s.strip()]; n_sents = max(len(sents), 1)
    n_chars = max(len(text), 1); letters = max(sum(c.isalpha() for c in text), 1); unique = len(set(lower))
    return {
        "n_chars": len(text), "n_words": len(words), "n_sentences": n_sents,
        "avg_word_len": float(np.mean([len(w) for w in words])) if words else 0.0,
        "long_word_ratio": sum(len(w) >= 7 for w in words) / n_words,
        "avg_sentence_len": n_words / n_sents,
        "commas_per_100w": text.count(",") / n_words * 100,
        "questions_per_100w": text.count("?") / n_words * 100,
        "exclaims_per_100w": text.count("!") / n_words * 100,
        "quotes_per_100w": text.count('"') / n_words * 100,
        "upper_ratio": sum(c.isupper() for c in text) / letters,
        "digit_ratio": sum(c.isdigit() for c in text) / n_chars,
        "lowercase_i_per_100w": len(LOWER_I_RE.findall(text)) / n_words * 100,
        "all_caps_word_ratio": sum(w.isupper() and len(w) > 1 for w in words) / n_words,
        "n_unique_words": unique, "ttr": unique / n_words, "root_ttr": unique / np.sqrt(n_words)}

# ---------- syntactic features (spaCy) ----------
nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
SUBORDINATE_DEPS = {"advcl", "ccomp", "acl", "relcl", "csubj", "xcomp"}
POS_GROUPS = {"NOUN": "noun", "VERB": "verb", "ADJ": "adj", "ADV": "adv", "PRON": "pron",
              "ADP": "prep", "AUX": "aux", "DET": "det", "PROPN": "propn"}
def syntactic_features(doc):
    alpha = [t for t in doc if t.is_alpha]; n_words = max(len(alpha), 1)
    sents = [s for s in doc.sents if any(t.is_alpha for t in s)]; n_sents = max(len(sents), 1)
    lens = np.array([sum(t.is_alpha for t in s) for s in sents] or [0])
    pos_counts = Counter(t.pos_ for t in alpha); dep_counts = Counter(t.dep_ for t in doc)
    depths = [max((len(list(t.ancestors)) for t in s), default=0) for s in sents]
    no_verb = [not any(t.pos_ in ("VERB", "AUX") for t in s) for s in sents]
    lower_start = [next((c for c in s.text if c.isalpha()), "").islower() for s in sents]
    feats = {
        "sp_n_sentences": len(sents), "sp_sent_len_mean": float(lens.mean()), "sp_sent_len_std": float(lens.std()),
        "sp_sent_len_max": int(lens.max()), "sp_short_sent_share": float((lens < 5).mean()),
        "sp_long_sent_share": float((lens > 40).mean()),
        "sp_clauses_per_sent": sum(dep_counts[d] for d in SUBORDINATE_DEPS) / n_sents,
        "sp_coord_per_sent": dep_counts["conj"] / n_sents,
        "sp_passive_per_100w": (dep_counts["nsubjpass"] + dep_counts["auxpass"]) / n_words * 100,
        "sp_tree_depth_mean": float(np.mean(depths)) if depths else 0.0, "sp_tree_depth_max": max(depths, default=0),
        "sp_no_verb_share": float(np.mean(no_verb)) if no_verb else 0.0,
        "sp_lower_start_share": float(np.mean(lower_start)) if lower_start else 0.0}
    for tag, name in POS_GROUPS.items():
        feats[f"pos_{name}"] = pos_counts[tag] / n_words
    return feats

# ---------- spelling + readability ----------
EXTRA_WORDS = ["driverless", "nasa", "facs", "unrra", "texting", "etc", "tv", "gpa", "webcam"]
CONTRACTION_ERRORS = {"dont", "doesnt", "didnt", "cant", "wont", "isnt", "wasnt", "arent", "werent", "wouldnt",
                      "couldnt", "shouldnt", "hasnt", "havent", "hadnt", "thats", "theres", "whats", "im", "ive",
                      "youre", "theyre"}
spell = SpellChecker(); spell.word_frequency.load_words(EXTRA_WORDS)
TOK_RE = re.compile(r"[A-Za-z][A-Za-z']*")
def _safe(func, text):
    try: return float(func(text))
    except Exception: return 0.0
def spelling_readability_features(text):
    t2 = text.replace("’", "'").replace("‘", "'")
    tokens = TOK_RE.findall(t2); n_words = max(len(tokens), 1)
    cands = [w.lower() for w in tokens if "'" not in w and len(w) > 1 and not (w[0].isupper() and not w.isupper())]
    unknown = spell.unknown(set(cands)); errors = [w for w in cands if w in unknown]
    n0 = max(len(TOK_RE.findall(text)), 1)
    return {"spell_errors": len(errors), "spell_err_per_100w": len(errors) / n_words * 100,
            "spell_unique_errors": len(set(errors)), "spell_err_share": len(errors) / max(len(cands), 1),
            "contraction_err_per_100w": sum(1 for w in tokens if w.lower() in CONTRACTION_ERRORS) / n_words * 100,
            "flesch_ease": _safe(textstat.flesch_reading_ease, text),
            "flesch_kincaid_grade": _safe(textstat.flesch_kincaid_grade, text),
            "gunning_fog": _safe(textstat.gunning_fog, text),
            "ari": _safe(textstat.automated_readability_index, text),
            "dale_chall": _safe(textstat.dale_chall_readability_score, text),
            "syllables_per_word": _safe(textstat.syllable_count, text) / n0}

# ---------- LanguageTool ----------
lt = language_tool_python.LanguageTool("en-US")
LT_CATS = {"GRAMMAR": "lt_grammar", "PUNCTUATION": "lt_punct", "TYPOGRAPHY": "lt_typography", "CASING": "lt_casing",
           "CONFUSED_WORDS": "lt_confused", "STYLE": "lt_style", "MISC": "lt_misc", "TYPOS": "lt_typos"}
def lt_features(text, matches):
    counts = Counter(str(getattr(m, "category", "")).upper() for m in matches)
    n_words = max(len(text.split()), 1)
    row = {name: counts.get(cat, 0) / n_words * 100 for cat, name in LT_CATS.items()}
    row["lt_nontypo_total"] = sum(v for k, v in counts.items() if k != "TYPOS") / n_words * 100
    return row

# ---------- CoLA sentence grammar scores ----------
SPLIT = re.compile(r"(?<=[.!?])\s+|\n+")
def split_sentences(text): return [s.strip() for s in SPLIT.split(text) if len(s.split()) >= 3]
def cola_encode(s):
    return [CLS] + tok(s.lower(), add_special_tokens=False, truncation=True, max_length=62)["input_ids"] + [SEP]
@torch.no_grad()
def cola_scores(sents):
    out = []
    for i in range(0, len(sents), 128):
        ids, att = tensors([cola_encode(s) for s in sents[i:i + 128]])
        with torch.autocast("cuda", dtype=torch.float16):
            logits = cola_model(input_ids=ids, attention_mask=att).logits
        out.append(torch.softmax(logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out) if out else np.array([])
def cola_aggregate(p):
    if len(p) == 0:
        return {"cola_mean": 0.5, "cola_min": 0.5, "cola_p25": 0.5, "cola_bad_share": 0.0, "cola_n_sent": 0}
    return {"cola_mean": float(p.mean()), "cola_min": float(p.min()), "cola_p25": float(np.quantile(p, 0.25)),
            "cola_bad_share": float((p < 0.5).mean()), "cola_n_sent": int(len(p))}

def compute_features(text):
    text = fix_text(normalize(text))
    matches = lt.check(text)
    sents = split_sentences(text); p = cola_scores(sents)
    feats = {}
    feats.update(surface_lexical_features(text)); feats.update(syntactic_features(nlp(text)))
    feats.update(spelling_readability_features(text)); feats.update(lt_features(text, matches))
    feats.update(cola_aggregate(p))
    return text, feats, matches, sents, p
print("Pipeline ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 54.2 MB/s eta 0:00:00


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

[transformers] The tokenizer you are loading from '/root/.cache/huggingface/hub/models--microsoft--deberta-v3-base/snapshots/8ccc9b6f36199bec6961081d44eb72fb3f7353f3' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


DeBERTa essay models: 5 | CoLA model loaded


Pipeline ready.


In [2]:
@torch.no_grad()
def deberta_score(text):
    x, att = tensors([wrap(tok(text, add_special_tokens=False)["input_ids"])])
    preds = []
    for m in essay_models:
        with torch.autocast("cuda", dtype=torch.float16):
            preds.append(float(m(x, att).float().cpu().numpy()[0, 0]))
    return float(np.mean(preds))

def score_essay(raw_text):
    text, feats, matches, sents, p = compute_features(raw_text)
    X_aes, X_ell = pd.DataFrame([feats])[AES_FEATS], pd.DataFrame([feats])[ELL_FEATS]
    lg, deb = float(aes_booster.predict(X_aes)[0]), deberta_score(text)
    blend = W * deb + (1 - W) * lg
    traits = {t: float(np.clip(np.round(ell_boosters[t].predict(X_ell)[0] * 2) / 2, 1, 5)) for t in TRAITS}
    contrib = pd.Series(aes_booster.predict(X_aes, pred_contrib=True)[0][:-1], index=AES_FEATS).sort_values()
    issues = [{"category": str(getattr(m, "category", "")), "message": getattr(m, "message", ""),
               "context": getattr(m, "context", "")} for m in matches if str(getattr(m, "category", "")).upper() != "STYLE"]
    weakest = sorted(zip(p.tolist(), sents))[:3]
    return {"score": int(1 + np.searchsorted(np.sort(CUTS), blend)), "continuous": blend, "deberta": deb, "lightgbm": lg,
            "traits": traits, "issues": issues, "weakest_sentences": weakest,
            "pushed_down": contrib.head(5).round(2).to_dict(), "pushed_up": contrib.tail(5)[::-1].round(2).to_dict(),
            "n_words": len(text.split())}

# ---------- 1) do our features match the ones saved during training? ----------
saved = (pd.read_csv(find("aes_prepared.csv")).merge(pd.read_csv(find("aes_features.csv")), on="essay_id")
         .merge(pd.read_csv(find("aes_languagetool.csv")), on="essay_id").merge(pd.read_csv(find("aes_cola.csv")), on="essay_id"))
sample = saved.sample(25, random_state=0).reset_index(drop=True)
mine = pd.DataFrame([compute_features(t)[1] for t in sample["text"]])[AES_FEATS]
diff = (mine - sample[AES_FEATS]).abs()
rep = pd.DataFrame({"max abs diff": diff.max(), "mean rel diff": (diff / (sample[AES_FEATS].abs() + 1e-6)).mean()})
bad = rep[rep["mean rel diff"] > 0.01]
print(f"{len(AES_FEATS) - len(bad)} of {len(AES_FEATS)} features match the saved values (mean relative difference <= 1%)")
print(bad.round(4).to_string() if len(bad) else "No mismatches.")

# ---------- 2) score a few training essays and one hand-written text ----------
for i in range(3):
    r = score_essay(sample["text"][i])
    print(f"\nEssay {sample['essay_id'][i]}: true score {sample['score'][i]} | predicted {r['score']} "
          f"(blend {r['continuous']:.2f}, DeBERTa {r['deberta']:.2f}, LightGBM {r['lightgbm']:.2f}) | {r['n_words']} words")
print("\nHand-written example:")
demo = ("I think that cars is bad for the enviroment. Everyone dont walk anymore and thats a problem! "
        "Many people drive there car to every place, even when the shop is close. If we use less cars, the air "
        "will be cleaner and we will be healthy. In conclusion, i believe we should limit cars.")
r = score_essay(demo)
print({k: r[k] for k in ["score", "continuous", "n_words", "traits", "pushed_down", "pushed_up"]})
print("Issues found:", [(x["category"], x["message"][:60]) for x in r["issues"][:6]])
print("Weakest sentences:", [(round(a, 2), b) for a, b in r["weakest_sentences"]])

64 of 64 features match the saved values (mean relative difference <= 1%)
No mismatches.

Essay d550b2d: true score 2 | predicted 2 (blend 2.17, DeBERTa 2.03, LightGBM 2.41) | 269 words

Essay 0c10954: true score 2 | predicted 2 (blend 2.24, DeBERTa 2.36, LightGBM 2.02) | 213 words

Essay ef04816: true score 3 | predicted 4 (blend 3.41, DeBERTa 3.44, LightGBM 3.36) | 426 words

Hand-written example:
{'score': 1, 'continuous': 1.4676174991816717, 'n_words': 54, 'traits': {'cohesion': 2.5, 'syntax': 2.5, 'vocabulary': 2.0, 'phraseology': 2.0, 'grammar': 2.5, 'conventions': 2.5}, 'pushed_down': {'n_chars': -0.55, 'ttr': -0.22, 'n_words': -0.19, 'cola_mean': -0.11, 'lt_typos': -0.07}, 'pushed_up': {'pos_propn': 0.06, 'commas_per_100w': 0.06, 'digit_ratio': 0.05, 'sp_tree_depth_max': 0.05, 'flesch_kincaid_grade': 0.03}}
Issues found: [('TYPOS', 'Possible spelling mistake found.'), ('TYPOS', 'Possible spelling mistake found.'), ('TYPOS', 'Possible spelling mistake found.'), ('GRAMMAR', 'Did 

In [3]:
import gradio as gr

def bar(v): 
    n = int(round((v - 1) / 4 * 10))
    return "█" * n + "░" * (10 - n)

def render(r):
    out = [f"## Holistic score: **{r['score']} / 6**  (continuous value {r['continuous']:.2f})"]
    if r["n_words"] < 150:
        out.append("> ⚠️ Fewer than 150 words: the model was trained on essays of 150+ words, so treat this score as a rough guide.")
    if r["n_words"] > 700:
        out.append("> ℹ️ Long essay: only the beginning and the end are read by the neural model (the middle is skipped).")
    out.append(f"\n*DeBERTa says {r['deberta']:.2f}, the feature model says {r['lightgbm']:.2f}; they are blended 65/35.*")
    out.append("\n### Writing traits (scale 1 to 5)\n| Trait | Score | |\n|---|---|---|")
    for t, v in r["traits"].items():
        out.append(f"| {t.capitalize()} | {v:.1f} | `{bar(v)}` |")
    out.append("\n### Grammar and spelling issues found")
    shown = r["issues"][:8]
    out += [f"- **{i['category'].title()}**: {i['message'][:90]}  \n  `{i['context'].strip()[:110]}`" for i in shown] or ["- None found."]
    if len(r["issues"]) > 8:
        out.append(f"- ...and {len(r['issues']) - 8} more.")
    out.append("\n### Least grammatical sentences (grammar judge, 0 = broken, 1 = fine)")
    weak = [(p, s) for p, s in r["weakest_sentences"] if p < 0.5]
    out += [f"- ({p:.2f}) {s}" for p, s in weak] or ["- No clearly ungrammatical sentences."]
    down = ", ".join(f"{k} ({v:+.2f})" for k, v in list(r["pushed_down"].items())[:3])
    up = ", ".join(f"{k} ({v:+.2f})" for k, v in list(r["pushed_up"].items())[:3])
    out.append(f"\n### What moved the feature model's score\n- Down: {down}\n- Up: {up}\n\n*Length features dominate this view; small values are noise.*")
    return "\n".join(out)

def run(text):
    if not text or len(text.split()) < 5:
        return "Please paste an essay (at least a few sentences)."
    return render(score_essay(text))

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive") == "Batch":
    examples = {"hand_written": demo, "training_essay": sample["text"][2]}
    json.dump({k: render(score_essay(v)) for k, v in examples.items()}, open("/kaggle/working/demo_examples.json", "w"), indent=1)
    print("Committed run: app skipped, example outputs saved to demo_examples.json")
else:
    with gr.Blocks(title="Automated Essay Scoring") as app:
        gr.Markdown("# Automated Essay Scoring\nPaste an essay to get a holistic score (1-6), six writing-trait scores, "
                    "grammar/spelling issues and the least grammatical sentences.")
        box = gr.Textbox(lines=14, label="Essay text", placeholder="Paste an essay here...")
        btn = gr.Button("Score essay", variant="primary")
        out = gr.Markdown()
        btn.click(run, inputs=box, outputs=out)
    app.launch(share=True, inline=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://57656cf05ba4a01396.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
